In [1]:
# ------------------- Task 01 ------------------- #


import math

class Node:
	def __init__(self, name, value=None):
		self.name = name
		self.value = value
		self.children = []
		self.minimax_value = None

class Environment:
	def __init__(self):
		self.root = Node('Root')
		zone_a = Node('Zone A')
		zone_b = Node('Zone B')
		self.root.children = [zone_a, zone_b]
		# Thief's options
		zone_a.children = [Node('A-Option1', 3), Node('A-Option2', 5)]
		zone_b.children = [Node('B-Option1', 2), Node('B-Option2', 9)]

	def get_percept(self, node):
		return node

	def print_states(self):
		print("All possible states and moves:")
		for zone in self.root.children:
			for thief in zone.children:
				print(f"Robot -> {zone.name}, Thief -> {thief.name.split('-')[1]}, Payoff: {thief.value}")

class MinimaxAgent:
	def __init__(self, depth):
		self.depth = depth
		self.computed_nodes = []

	def compute_minimax(self, node, depth, maximizing):
		if depth == 0 or not node.children:
			self.computed_nodes.append(node.name)
			return node.value
		if maximizing:
			value = -math.inf
			for child in node.children:
				value = max(value, self.compute_minimax(child, depth-1, False))
			node.minimax_value = value
			self.computed_nodes.append(node.name)
			return value
		else:
			value = math.inf
			for child in node.children:
				value = min(value, self.compute_minimax(child, depth-1, True))
			node.minimax_value = value
			self.computed_nodes.append(node.name)
			return value

	def act(self, percept, environment):
		print("\nMinimax computation:")
		best_value = -math.inf
		best_move = None
		for zone in percept.children:
			value = self.compute_minimax(zone, self.depth-1, False)
			print(f"If Robot chooses {zone.name}, Thief minimizes to: {value}")
			if value > best_value:
				best_value = value
				best_move = zone.name
		print(f"\nRobot should choose: {best_move} (guaranteed payoff: {best_value})")

def run_agent(agent, environment, start_node):
	percept = environment.get_percept(start_node)
	agent.act(percept, environment)

environment = Environment()
environment.print_states()
agent = MinimaxAgent(depth=2)
run_agent(agent, environment, environment.root)



All possible states and moves:
Robot -> Zone A, Thief -> Option1, Payoff: 3
Robot -> Zone A, Thief -> Option2, Payoff: 5
Robot -> Zone B, Thief -> Option1, Payoff: 2
Robot -> Zone B, Thief -> Option2, Payoff: 9

Minimax computation:
If Robot chooses Zone A, Thief minimizes to: 3
If Robot chooses Zone B, Thief minimizes to: 2

Robot should choose: Zone A (guaranteed payoff: 3)


In [2]:
# ------------------- Task 02 ------------------- #

import math
class Node:
	def __init__(self, name, value=None):
		self.name = name
		self.value = value
		self.children = []
		self.minimax_value = None

# Minimax algorithm implementation
class Minimax:
	def __init__(self):
		self.visited = []

	def minimax(self, node, maximizing_player=True):
		if not node.children:
			self.visited.append(node.name)
			node.minimax_value = node.value
			return node.value
		if maximizing_player:
			value = -math.inf
			for child in node.children:
				value = max(value, self.minimax(child, False))
			node.minimax_value = value
			self.visited.append(node.name)
			return value
		else:
			value = math.inf
			for child in node.children:
				value = min(value, self.minimax(child, True))
			node.minimax_value = value
			self.visited.append(node.name)
			return value

# Alpha-Beta Pruning implementation
class AlphaBeta:
	def __init__(self):
		self.visited = []
		self.pruned = []

	def alphabeta(self, node, alpha=-math.inf, beta=math.inf, maximizing_player=True):
		if not node.children:
			self.visited.append(node.name)
			node.minimax_value = node.value
			return node.value
		if maximizing_player:
			value = -math.inf
			for child in node.children:
				v = self.alphabeta(child, alpha, beta, False)
				value = max(value, v)
				alpha = max(alpha, value)
				if beta <= alpha:
					# Prune remaining children
					idx = node.children.index(child)
					for pruned_child in node.children[idx+1:]:
						self.pruned.append(pruned_child.name)
					break
			node.minimax_value = value
			self.visited.append(node.name)
			return value
		else:
			value = math.inf
			for child in node.children:
				v = self.alphabeta(child, alpha, beta, True)
				value = min(value, v)
				beta = min(beta, value)
				if beta <= alpha:
					idx = node.children.index(child)
					for pruned_child in node.children[idx+1:]:
						self.pruned.append(pruned_child.name)
					break
			node.minimax_value = value
			self.visited.append(node.name)
			return value

root = Node('Start')
stop = Node('Stop')
go = Node('Go')
turn = Node('Turn')
root.children = [stop, go, turn]

stop.children = [Node('Stop-1', 6), Node('Stop-2', 7)]
go.children = [Node('Go-1', 2), Node('Go-2', 4)]
turn.children = [Node('Turn-1', 8), Node('Turn-2', 3)]

# --- Minimax ---
minimax_solver = Minimax()
best_value = minimax_solver.minimax(root, maximizing_player=True)

print("Minimax Traversal Order:", minimax_solver.visited)
print("Minimax Value at Root (Best Action Value):", best_value)
print("Action Values:")
for action in root.children:
	print(f"{action.name}: {action.minimax_value}")

# --- Alpha-Beta Pruning ---
alphabeta_solver = AlphaBeta()
ab_value = alphabeta_solver.alphabeta(root, maximizing_player=True)

print("\nAlpha-Beta Traversal Order:", alphabeta_solver.visited)
print("Alpha-Beta Value at Root (Best Action Value):", ab_value)
print("Action Values:")
for action in root.children:
	print(f"{action.name}: {action.minimax_value}")

print("Pruned Branches:", alphabeta_solver.pruned)
print("Number of Nodes Not Evaluated due to Pruning:", len(alphabeta_solver.pruned))



Minimax Traversal Order: ['Stop-1', 'Stop-2', 'Stop', 'Go-1', 'Go-2', 'Go', 'Turn-1', 'Turn-2', 'Turn', 'Start']
Minimax Value at Root (Best Action Value): 6
Action Values:
Stop: 6
Go: 2
Turn: 3

Alpha-Beta Traversal Order: ['Stop-1', 'Stop-2', 'Stop', 'Go-1', 'Go', 'Turn-1', 'Turn-2', 'Turn', 'Start']
Alpha-Beta Value at Root (Best Action Value): 6
Action Values:
Stop: 6
Go: 2
Turn: 3
Pruned Branches: ['Go-2']
Number of Nodes Not Evaluated due to Pruning: 1


In [3]:
# ------------------- Task 03 ------------------- #

import math

class Node:
	def __init__(self, name=None, value=None):
		self.name = name
		self.value = value
		self.children = []
		self.minimax_value = None

def build_tree():
	root = Node('Root')
	attack = Node('Attack')
	defend = Node('Defend')
	gather = Node('Gather')
	root.children = [attack, defend, gather]

	a1 = Node('A1')
	a2 = Node('A2')
	d1 = Node('D1')
	d2 = Node('D2')
	g1 = Node('G1')
	g2 = Node('G2')
	attack.children = [a1, a2]
	defend.children = [d1, d2]
	gather.children = [g1, g2]

	a1.children = [Node(None, 5), Node(None, 6)]
	a2.children = [Node(None, 7), Node(None, 4)]
	d1.children = [Node(None, 3), Node(None, 8)]
	d2.children = [Node(None, 6), Node(None, 2)]
	g1.children = [Node(None, 1), Node(None, 9)]
	g2.children = [Node(None, 4), Node(None, 7)]

	return root

# Minimax algorithm
def minimax(node, depth, maximizing_player):
	if depth == 0 or not node.children:
		return node.value
	if maximizing_player:
		value = -math.inf
		for child in node.children:
			value = max(value, minimax(child, depth-1, False))
		node.minimax_value = value
		return value
	else:
		value = math.inf
		for child in node.children:
			value = min(value, minimax(child, depth-1, True))
		node.minimax_value = value
		return value

# Alpha-Beta Pruning algorithm
def alpha_beta(node, depth, alpha, beta, maximizing_player, ab_trace=None, pruned=None):
	if ab_trace is not None:
		ab_trace.append((node.name, alpha, beta))
	if depth == 0 or not node.children:
		return node.value
	if maximizing_player:
		value = -math.inf
		for child in node.children:
			value = max(value, alpha_beta(child, depth-1, alpha, beta, False, ab_trace, pruned))
			alpha = max(alpha, value)
			if beta <= alpha:
				if pruned is not None:
					pruned.append(child.name)
				break
		node.minimax_value = value
		return value
	else:
		value = math.inf
		for child in node.children:
			value = min(value, alpha_beta(child, depth-1, alpha, beta, True, ab_trace, pruned))
			beta = min(beta, value)
			if beta <= alpha:
				if pruned is not None:
					pruned.append(child.name)
				break
		node.minimax_value = value
		return value

def count_nodes(node):
	if not node.children:
		return 1
	return 1 + sum(count_nodes(child) for child in node.children)

root = build_tree()
depth = 3

# a) Minimax
minimax_value = minimax(root, depth, True)
print(f"a) Minimax value at root: {minimax_value}")
print(f"   Best action for MAX: ", end='')
for child in root.children:
    if child.minimax_value == minimax_value:
        print(child.name)

# b) Alpha-Beta Pruning
ab_trace = []
pruned = []
root_ab = build_tree()  # fresh tree for AB
ab_value = alpha_beta(root_ab, depth, -math.inf, math.inf, True, ab_trace, pruned)
print(f"\nb) Alpha-Beta value at root: {ab_value}")
print("   Alpha-Beta trace (node, alpha, beta):")
for entry in ab_trace:
    print(f"   {entry}")

# c) Pruned branches
print(f"\nc) Pruned branches: {pruned}")

# d) Node counts
print(f"\nd) Nodes evaluated in Minimax: {count_nodes(root)}")
print(f"   Nodes evaluated in Alpha-Beta: {len(ab_trace)}")

# e) Explanation
print("\ne) Alpha-Beta Pruning improves efficiency by pruning branches that cannot affect the final decision, reducing the number of nodes evaluated compared to standard Minimax.")


a) Minimax value at root: 7
   Best action for MAX: Gather

b) Alpha-Beta value at root: 7
   Alpha-Beta trace (node, alpha, beta):
   ('Root', -inf, inf)
   ('Attack', -inf, inf)
   ('A1', -inf, inf)
   (None, -inf, inf)
   (None, 5, inf)
   ('A2', -inf, 6)
   (None, -inf, 6)
   ('Defend', 6, inf)
   ('D1', 6, inf)
   (None, 6, inf)
   (None, 6, inf)
   ('D2', 6, 8)
   (None, 6, 8)
   (None, 6, 8)
   ('Gather', 6, inf)
   ('G1', 6, inf)
   (None, 6, inf)
   (None, 6, inf)
   ('G2', 6, 9)
   (None, 6, 9)
   (None, 6, 9)

c) Pruned branches: [None, 'D2']

d) Nodes evaluated in Minimax: 22
   Nodes evaluated in Alpha-Beta: 21

e) Alpha-Beta Pruning improves efficiency by pruning branches that cannot affect the final decision, reducing the number of nodes evaluated compared to standard Minimax.
